# BillFlow YOLOv8 Training
Train product detector for on-device inference via TFLite.
Works on: local RTX 4050 · Google Colab T4 · any CUDA GPU.

**Pre-requisites:**
- Run `open-food-facts-scraper.ts` + `google-images-scraper.py`
- Annotate images in Label Studio and export YOLO labels
- Run `prepare-dataset.ts` to build `datasets/billflow/`

In [ ]:
# Cell 1: Install dependencies
# Skip if already installed (comment out after first run)
%pip install ultralytics>=8.2.0 --quiet
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --quiet
%pip install opencv-python-headless pillow --quiet

import ultralytics
ultralytics.checks()  # Prints versions + environment summary

In [ ]:
# Cell 2: Verify GPU availability
import torch
import subprocess

print('=== GPU Info ===')
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout[:2000])
except FileNotFoundError:
    print('nvidia-smi not found (running on CPU or non-NVIDIA GPU)')

print('\n=== PyTorch CUDA ===')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device:         {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    device = 0  # GPU index
else:
    print('WARNING: No CUDA GPU. Training will be slow on CPU.')
    device = 'cpu'

print(f'\nTraining device: {device}')

In [ ]:
# Cell 3: Mount dataset
# Option A — Local path (RTX 4050 local training)
import os
from pathlib import Path

# Adjust path if running notebook from a different directory
DATASET_YAML = Path('../../datasets/billflow/billflow.yaml').resolve()

# Option B — Google Colab: uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_YAML = Path('/content/drive/MyDrive/billflow/datasets/billflow/billflow.yaml')

assert DATASET_YAML.exists(), f'Dataset YAML not found: {DATASET_YAML}\nRun prepare-dataset.ts first.'
print(f'Dataset: {DATASET_YAML}')

# Count images per split
for split in ['train', 'val', 'test']:
    img_dir = DATASET_YAML.parent / 'images' / split
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    lbl_dir = DATASET_YAML.parent / 'labels' / split
    lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
    print(f'  {split:6}: {len(imgs):5} images, {len(lbls):5} labels')

In [ ]:
# Cell 4: Train YOLOv8
# Model options (smallest → largest, accuracy vs speed):
#   yolov8n.pt  — nano  (~3.2M params) — best for mobile / TFLite
#   yolov8s.pt  — small (~11M params)
#   yolov8m.pt  — medium
#
# RTX 4050 (6GB VRAM): batch=16, imgsz=640 works fine with yolov8n
# If CUDA OOM: reduce batch to 8 or imgsz to 416

from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # Downloads pretrained weights on first run

results = model.train(
    data=str(DATASET_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    project='runs/detect',
    name='billflow_v1',
    # Augmentation (helps small datasets)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    # Early stopping
    patience=20,
    # Save best weights
    save=True,
    save_period=10,
    # Logging
    verbose=True,
)

print(f'\nBest weights: {results.save_dir}/weights/best.pt')

In [ ]:
# Cell 5: Validate best model
from ultralytics import YOLO
from pathlib import Path

best_weights = Path('runs/detect/billflow_v1/weights/best.pt')
assert best_weights.exists(), 'Run Cell 4 first'

model = YOLO(str(best_weights))

metrics = model.val(
    data=str(DATASET_YAML),
    imgsz=640,
    batch=16,
    device=device,
    split='val',
)

print('\n=== Validation Results ===')
print(f'mAP@50:     {metrics.box.map50:.3f}')
print(f'mAP@50-95:  {metrics.box.map:.3f}')
print(f'Precision:  {metrics.box.mp:.3f}')
print(f'Recall:     {metrics.box.mr:.3f}')
print('\nPer-class mAP50:')
for cls_name, ap in zip(metrics.names.values(), metrics.box.ap50):
    print(f'  {cls_name:30}: {ap:.3f}')

In [ ]:
# Cell 6: Export to TFLite (int8 quantized for on-device inference)
# int8 quantization: ~4× smaller model, ~2× faster on mobile, minimal accuracy loss
from ultralytics import YOLO
from pathlib import Path

best_weights = Path('runs/detect/billflow_v1/weights/best.pt')
model = YOLO(str(best_weights))

# Export to TFLite with int8 quantization
# imgsz must match training imgsz
export_path = model.export(
    format='tflite',
    imgsz=640,
    int8=True,       # Quantize weights to int8
    nms=True,        # Include NMS in TFLite graph (simpler Flutter integration)
    dynamic=False,   # Fixed input size for mobile
)

print(f'\nTFLite model: {export_path}')

# Check file size
size_mb = Path(export_path).stat().st_size / 1e6
print(f'Model size:   {size_mb:.1f} MB')
print('\nTarget: < 10 MB for comfortable inclusion in Flutter APK')

In [ ]:
# Cell 7: Copy TFLite model to Flutter assets
import shutil
from pathlib import Path

# Find exported .tflite file
tflite_src = Path('runs/detect/billflow_v1/weights/best_saved_model/best_float32.tflite')
# Try int8 path if float32 not found
if not tflite_src.exists():
    tflite_src = Path('runs/detect/billflow_v1/weights/best_int8.tflite')
if not tflite_src.exists():
    # Find any .tflite in weights dir
    candidates = list(Path('runs/detect/billflow_v1/weights').rglob('*.tflite'))
    assert candidates, 'No .tflite found. Run Cell 6 first.'
    tflite_src = candidates[0]

# Destination: flutter_app/assets/models/
# Adjust relative path if notebook is not in backend/scripts/training/
flutter_models = Path('../../../flutter_app/assets/models')
flutter_models.mkdir(parents=True, exist_ok=True)

dest = flutter_models / 'billflow_detector.tflite'
shutil.copy2(tflite_src, dest)

size_mb = dest.stat().st_size / 1e6
print(f'Copied: {tflite_src.name} → {dest}')
print(f'Size:   {size_mb:.1f} MB')
print()
print('Next steps:')
print('  1. Add to flutter_app/pubspec.yaml:')
print('       assets:')
print('         - assets/models/billflow_detector.tflite')
print('  2. Add tflite_flutter to pubspec.yaml dependencies')
print('  3. Load model in Flutter:')
print('       final interpreter = await Interpreter.fromAsset(')
print('           \'assets/models/billflow_detector.tflite\');')